# 02 Web后端开发 · 第03课：Pydantic数据模型

上一课使用`dict[str, Any]`接收JSON请求体。字典虽然灵活，但不能直接说明必须有哪些字段、每个字段是什么类型、哪些字段可以不传。Pydantic用模型把这些数据规则写成代码，FastAPI会据此自动验证请求并生成接口文档。

## 本课目标

1. 理解Pydantic模型解决什么问题；
2. 使用`BaseModel`声明数据字段；
3. 区分必填字段、默认值字段和可空字段；
4. 使用`Field`添加长度和数值限制；
5. 访问模型属性并使用`model_dump()`转换数据；
6. 理解FastAPI如何自动验证请求体并返回422；
7. 区分请求模型和响应模型；
8. 使用`response_model`约束和过滤响应数据。

## 运行本课前：安装必要依赖

本课用到的第三方库已统一写在同目录的`requirements.txt`中。你只需在下面两种安装方式中选一种，不用两种都执行。

### 方式A：在PowerShell终端中安装（推荐）

先进入本课目录：

```powershell
Set-Location "D:\桌面\学点东西\codex—help\02_Web后端开发\03_Pydantic数据模型"
```

再安装依赖：

```powershell
python -m pip install -r requirements.txt
```

这条命令的每一部分：

- `python`：使用当前终端环境中的Python；
- `-m pip`：让这个Python运行自己对应的`pip`，减少“包安装到另一个Python”的问题；
- `install`：执行安装；
- `-r requirements.txt`：读取文件中列出的所有依赖和版本范围。

`requirements.txt`中的包分别用于：

- `fastapi`：创建Web API；
- `pydantic`：定义和验证数据模型；
- `uvicorn[standard]`：在本地启动FastAPI服务；
- `httpx2`：为FastAPI的`TestClient`提供HTTP测试能力；
- `pytest`：运行自动化测试。

### 方式B：直接在Jupyter中安装

如果终端的Python和Notebook内核不是同一个环境，可以在Notebook新建一个代码单元，运行：

```python
%pip install -r "D:\桌面\学点东西\codex—help\02_Web后端开发\03_Pydantic数据模型\requirements.txt"
```

`%pip`是Jupyter/IPython提供的魔法命令，会尽量把包安装到当前Notebook使用的Python环境。安装完成后，如果Jupyter提示重启内核，请先重启内核再继续。

### 检查是否安装成功

在PowerShell中运行：

```powershell
python -m pip show fastapi pydantic uvicorn httpx2 pytest
```

如果能看到这些包的`Name`、`Version`和`Location`，就说明当前Python环境已经找到它们。

## 1. 为什么普通字典还不够？

假设接口收到：

```python
payload = {"username": "小明", "age": 20}
```

仅仅看到参数类型`dict[str, Any]`时，FastAPI和阅读代码的人无法直接知道：

- `username`是不是必填；
- `age`必须是整数还是也允许字符串；
- 用户名是否有长度限制；
- 是否允许缺少年龄；
- 错误数据应该怎样提示；
- 自动文档应该展示什么请求结构。

如果全部手动检查，代码会出现很多重复判断。Pydantic模型就是一份可执行的数据契约：它同时告诉程序、开发者和自动文档“数据应该长什么样”。

## 2. 第一个Pydantic模型

Pydantic模型使用Python类和类型标注声明字段。在写第一个模型之前，先弄清代码中最重要的`BaseModel`。

### 2.1 `BaseModel`到底是什么？

`BaseModel`是第三方库Pydantic提供的一个**类**，不是函数，也不是FastAPI接口。`Base`表示“基础”，因此它通常被称为**基类**或**父类**。

```python
from pydantic import BaseModel
```

这行代码的意思是：从`pydantic`库中导入名为`BaseModel`的类。

```python
class UserProfile(BaseModel):
    username: str
    age: int
```

在`class UserProfile(BaseModel)`中，括号里的`BaseModel`不是传给函数的参数。这是Python的**继承**语法，意思是：`UserProfile`继承`BaseModel`，成为一个Pydantic模型类。

可以把`BaseModel`想成一套“已经造好的智能表单底座”。你只需声明表单上有哪些字段，它就会帮你补上一系列能力：

1. **读取字段规则**：把`username: str`和`age: int`理解为数据要求；
2. **接收数据并创建对象**：允许使用`UserProfile(username=..., age=...)`创建实例；
3. **验证与转换**：检查必填字段和类型，并尝试把可合理转换的值转成目标类型；
4. **提供模型方法和描述**：例如后面会学的`model_dump()`，以及FastAPI用来生成接口文档的字段信息。

`BaseModel`不会把数据保存进数据库，也不会自己发送HTTP请求。它的核心工作是：**把输入数据按照你声明的规则，变成经过验证的Python对象。**

创建模型实例时，可以先把执行流程理解为：

```text
传入username和age
        ↓
BaseModel读取UserProfile的字段声明
        ↓
检查必填字段、类型和限制
        ↓
验证成功：得到UserProfile对象
验证失败：产生验证错误（第4节会详细讲）
```

In [ ]:
# BaseModel类来自第三方库pydantic
# 继承BaseModel后，Pydantic会读取类中的字段类型并进行数据验证
from pydantic import BaseModel


class UserProfile(BaseModel):
    username: str
    age: int


# UserProfile(username=..., age=...)：创建并验证一个模型实例
#   username参数对应模型中的username字段
#   age参数对应模型中的age字段
#   返回值：通过验证的UserProfile对象
user = UserProfile(username="小明", age=20)

print(user)
print(type(user))
print(user.username)
print(user.age)

### 对比：不继承`BaseModel`时有什么不同？

下面的`PlainUser`是一个最普通的Python类。它虽然也写了`username: str`和`age: int`，但这些类型标注本身不会自动执行Pydantic式的验证。

In [ ]:
class PlainUser:
    username: str
    age: int


# 最普通且没有自己定义__init__的类，需要先空着创建对象
plain_user = PlainUser()
plain_user.username = "小明"
plain_user.age = "二十"  # 虽然声明了int，普通类仍不会在这里自动拒绝字符串

print(plain_user.age)
print(type(plain_user.age))

# UserProfile继承了BaseModel，创建对象时会读取和验证字段
checked_user = UserProfile(username="小明", age=20)
print(checked_user.age)
print(type(checked_user.age))

对比结果：

- 普通类的类型标注主要是给阅读者和检查工具看，不会自动变成完整的运行时验证器；
- 继承`BaseModel`后，Pydantic会读取这些标注，在创建模型对象时执行验证。

<details>
<summary>小检查：如果把class UserProfile(BaseModel)改成class UserProfile，会失去什么？</summary>

它就不再是Pydantic模型，会失去Pydantic提供的初始化、数据验证、合理类型转换、`model_dump()`等能力。FastAPI也不能再把它当成标准的Pydantic请求模型使用。

</details>

`UserProfile`是我们定义的模型类，`user`是这个类的实例。模型实例不是普通字典，因此读取字段通常使用属性：

```python
user.username
```

而不是：

```python
user["username"]
```

Pydantic会在创建模型实例时执行验证。只有数据通过验证，后面的代码才能得到正常模型对象。

### 示例2：Pydantic可能进行合理的类型转换

HTTP和JSON中的数据不一定已经是最终Python类型。Pydantic默认会尝试进行合理转换：

In [ ]:
# age收到的是字符串"21"，但模型声明age必须是int
converted_user = UserProfile(username="小红", age="21")

print(converted_user.age)
print(type(converted_user.age))

# 预期：值是21，类型是int

Pydantic不是简单地执行`isinstance()`后立即拒绝所有不同类型。它会先尝试解析；`"21"`可以解析成整数，但`"二十一"`不能。

### 课堂小练习1：声明你的第一个模型

**目标：**练习继承`BaseModel`、声明字段、创建实例和读取属性。

请定义`CourseInfo`模型，包含必填字符串`name`和必填整数`lesson_count`。创建一个名称为`Web后端`、课数为`6`的实例，并打印两个属性。

**预期结果：**打印`Web后端`和整数`6`。

<details>
<summary>完成后查看参考答案</summary>

```python
class CourseInfo(BaseModel):
    name: str
    lesson_count: int

course = CourseInfo(name="Web后端", lesson_count=6)
print(course.name)
print(course.lesson_count)
```

自检时再试一次`lesson_count="6"`，观察它是否被转换成整数。

</details>

In [ ]:
# 课堂小练习1：请在这里编写CourseInfo模型并创建实例
# 完成后运行本单元，对照上方预期结果自检


## 3. 必填、默认值和可空不是一回事

观察下面三个字段：

In [ ]:
class Article(BaseModel):
    title: str
    published: bool = False
    summary: str | None = None


article = Article(title="Pydantic入门")
print(article)

| 声明 | 能否省略 | 能否传`None` | 含义 |
|---|---|---|---|
| `title: str` | 不能 | 不能 | 必填字符串 |
| `published: bool = False` | 可以 | 不能 | 省略时使用`False` |
| `summary: str \| None = None` | 可以 | 可以 | 省略或显式传`None`都可以 |

特别注意：

```python
summary: str | None
```

表示值可以是字符串或`None`，但因为没有默认值，这个字段本身仍然必须出现在输入中。

```python
summary: str | None = None
```

才同时表示“可以省略”和“可以为`None`”。

### 示例：同一个模型的四种输入

下面通过多种输入观察“省略”和“传`None`”的区别。

In [ ]:
# 只提供必填字段：其余字段使用默认值
article1 = Article(title="第一篇")

# 主动覆盖published默认值
article2 = Article(title="第二篇", published=True)

# summary允许显式传入None
article3 = Article(title="第三篇", summary=None)

# summary也可以传字符串
article4 = Article(title="第四篇", summary="简短摘要")

print(article1)
print(article2)
print(article3)
print(article4)

### 课堂小练习2：判断字段规则

**目标：**区分必填、默认值和可空。

请先不运行代码，判断下面哪些创建方式有效，再逐个运行验证：

```python
Article()
Article(title="新闻")
Article(title="新闻", published=None)
Article(title="新闻", summary=None)
```

**自检：**有效调用应能创建实例；无效调用会产生`ValidationError`。

<details>
<summary>完成后查看答案与原因</summary>

- `Article()`无效，因为缺少必填的`title`；
- `Article(title="新闻")`有效，其他字段使用默认值；
- `published=None`无效，因为`published`只能是布尔值；
- `summary=None`有效，因为`summary`允许`None`。

</details>

In [ ]:
# 课堂小练习2：把四个Article(...)调用逐个复制到这里运行
# 建议一次只运行一个可能失败的调用，方便观察对应错误


## 4. 验证失败时发生什么？

在普通Python代码中直接创建模型，验证失败会抛出`ValidationError`。

In [ ]:
# ValidationError类来自pydantic，表示模型数据验证失败
from pydantic import ValidationError


try:
    UserProfile(username="小明", age="不是数字")
except ValidationError as error:
    # error.errors()：取得结构化错误列表
    #   不需要参数
    #   返回值：由错误字典组成的列表
    print(error.errors())

错误信息通常会包含：

- 错误发生在哪个字段；
- 期望的数据类型或规则；
- 实际收到的输入；
- 错误类型。

在FastAPI接口中，FastAPI会捕获请求数据产生的验证错误，并自动转换成HTTP `422`响应，因此路由处理函数通常不需要自己捕获这类错误。

### 示例2：一次输入可能有多个错误

Pydantic会尽量收集各字段的错误，而不是只告诉你第一处失败。

In [ ]:
try:
    # username类型错误，同时age也无法转换成整数
    UserProfile(username=None, age="二十")
except ValidationError as error:
    for item in error.errors():
        print("位置：", item["loc"])
        print("说明：", item["msg"])
        print("输入：", item["input"])
        print("---")

这里的`item`是错误字典。`loc`表示错误字段位置，`msg`是说明，`input`是实际输入。这些字段由Pydantic错误结构定义，不是Python关键字。

### 课堂小练习3：读懂验证错误

**目标：**根据错误位置判断哪个字段不合法。

创建`UserProfile(username="小李", age="abc")`，捕获`ValidationError`并打印`error.errors()`。请找出：

1. 错误发生在哪个字段；
2. 实际输入是什么；
3. 为什么`"21"`可以通过，而`"abc"`不能。

<details>
<summary>完成后查看解释</summary>

错误位置是`age`，实际输入是`"abc"`。两者都是字符串，但`"21"`的内容可以解析成整数，`"abc"`不可以。

</details>

In [ ]:
# 课堂小练习3：在这里创建错误的UserProfile并捕获ValidationError
# 打印error.errors()，然后查找loc、input和msg


## 5. 使用`Field`添加字段限制

类型只能说明“这是字符串”或“这是数字”，还不能说明长度和数值范围。`Field`用于给单个模型字段增加规则和文档信息。

In [ ]:
# Field函数来自pydantic，不是fastapi.Field
from pydantic import Field


class ProductCreate(BaseModel):
    # Field(...)：声明该字段没有默认值，因此仍然必填
    #   第1个位置参数default：...表示必填字段
    #   min_length参数：字符串最少字符数
    #   max_length参数：字符串最多字符数
    #   description参数：显示在自动接口文档中的字段说明
    name: str = Field(
        ...,
        min_length=2,
        max_length=30,
        description="商品名称",
    )

    # gt参数表示greater than，即必须严格大于指定值
    price: float = Field(..., gt=0, description="商品价格")

    # ge表示greater than or equal，即大于等于
    # le表示less than or equal，即小于等于
    stock: int = Field(default=0, ge=0, le=10000)


product = ProductCreate(name="键盘", price=299.0)
print(product)

本课使用的约束参数：

| 参数 | 含义 |
|---|---|
| `min_length` | 最小长度 |
| `max_length` | 最大长度 |
| `gt` | 严格大于 |
| `ge` | 大于等于 |
| `lt` | 严格小于 |
| `le` | 小于等于 |
| `description` | 字段文档说明 |

`Field`描述的是数据契约，不是接口处理函数中的手动`if`。Pydantic会在模型创建阶段执行这些约束。

### 示例2：边界值与失败值

`min_length=2`表示长度等于2可以通过，长度1不能通过；`gt=0`表示价格必须严格大于0，因此0也不能通过。

In [ ]:
# 边界合法：name长度正好为2，price大于0，stock等于允许的下界0
boundary_product = ProductCreate(name="鼠标", price=0.01, stock=0)
print(boundary_product)

# 逐个测试错误，避免一个异常阻止后面的例子
invalid_products = [
    {"name": "鼠", "price": 10},       # 名称太短
    {"name": "鼠标", "price": 0},     # gt=0不允许等于0
    {"name": "鼠标", "price": 10, "stock": -1},  # ge=0不允许负数
]

for data in invalid_products:
    try:
        ProductCreate(**data)
    except ValidationError as error:
        print(data, "验证失败：", error.errors()[0]["msg"])

上面`ProductCreate(**data)`中的`**`是字典拆包：把字典键值作为关键字参数传入。它相当于根据字典内容调用`ProductCreate(name=..., price=..., stock=...)`。

### 课堂小练习4：设计受约束字段

**目标：**练习`Field`的字符串长度和数值范围。

定义`ExamResult`模型：

- `student`是必填字符串，长度2至20；
- `score`是必填整数，范围0至100（包含0和100）。

用`student="小王", score=100`自检，应创建成功；把`score`改成101，应产生`ValidationError`。

<details>
<summary>完成后查看参考答案</summary>

```python
class ExamResult(BaseModel):
    student: str = Field(..., min_length=2, max_length=20)
    score: int = Field(..., ge=0, le=100)
```

`ge=0`允许等于0，`le=100`允许等于100。

</details>

In [ ]:
# 课堂小练习4：请在这里定义ExamResult并测试score=100和score=101


## 6. 模型实例怎样转换成字典？

Pydantic v2使用`model_dump()`把模型实例转换成Python字典。

In [ ]:
# product.model_dump()：把当前Pydantic模型实例转换成Python字典
#   本例不传参数
#   返回值：包含模型字段和值的dict
product_data = product.model_dump()

print(product_data)
print(type(product_data))

方向要分清：

```text
ProductCreate(...)   输入数据 → Pydantic模型实例
product.model_dump() Pydantic模型实例 → Python字典
```

`model_dump()`是Pydantic v2的API。旧教程中可能看到`.dict()`，本课程统一使用当前v2写法。

### 示例2：决定是否保留值为`None`的字段

`model_dump()`还可以通过参数控制输出内容。

In [ ]:
article_without_summary = Article(title="没有摘要")

# model_dump(exclude_none=False)：False是默认值，保留值为None的字段
with_none = article_without_summary.model_dump()

# model_dump(exclude_none=True)：不把值为None的字段放进结果字典
#   exclude_none参数：是否排除值为None的字段
#   返回值：Python字典
without_none = article_without_summary.model_dump(exclude_none=True)

print(with_none)
print(without_none)

### 课堂小练习5：模型与字典之间的区别

**目标：**练习属性访问和`model_dump()`。

使用前面的`product`实例：

1. 用属性读取商品名称；
2. 把整个模型转换成字典；
3. 在转换后的字典中增加`category="外设"`；
4. 确认原模型没有自动增加`category`字段。

**预期结果：**修改转换后的字典不会改变原模型结构。

<details>
<summary>完成后查看参考代码</summary>

```python
print(product.name)
product_data = product.model_dump()
product_data["category"] = "外设"
print(product_data)
print(product)
```

</details>

In [ ]:
# 课堂小练习5：在这里把product转换成字典并增加category字段


## 7. FastAPI怎样使用Pydantic请求模型？

当路由函数的参数类型是Pydantic模型时，FastAPI会把它当作JSON请求体。

In [ ]:
from typing import Any

from fastapi import FastAPI


app = FastAPI(title="Pydantic示例")


class UserCreate(BaseModel):
    username: str = Field(..., min_length=2, max_length=20)
    age: int = Field(..., ge=1, le=120)
    bio: str | None = None
    password: str = Field(..., min_length=6)


@app.post("/users", status_code=201)
def create_user(user: UserCreate) -> dict[str, Any]:
    # FastAPI进入函数前已经完成：读取JSON、创建UserCreate实例、验证字段
    # 所以这里的user不是字典，而是验证通过的UserCreate对象
    data = user.model_dump()
    data["id"] = 1
    return data

客户端提交：

```json
{
  "username": "小明",
  "age": 20,
  "password": "secret123"
}
```

请求到达后的流程：

```text
JSON请求体
   ↓ FastAPI解析JSON
Python数据
   ↓ Pydantic创建UserCreate并验证
验证成功的user模型实例
   ↓ 才调用create_user(user)
路由函数返回数据
```

如果缺少`username`、年龄超出范围或密码太短，验证会在调用`create_user()`之前失败，FastAPI自动返回`422`。

### 可运行示例：用TestClient观察成功与422

下面不启动Uvicorn，直接使用上一课学过的`TestClient`调用FastAPI应用。

In [ ]:
# TestClient来自fastapi.testclient，用于在Python进程内模拟客户端
from fastapi.testclient import TestClient

# TestClient(app)：app参数是要测试的FastAPI应用对象
# 返回值是TestClient对象
client = TestClient(app)

# client.post(url, json=data)：发送模拟POST请求
#   url参数：请求路径
#   json参数：要编码成JSON请求体的Python对象
#   返回值：Response响应对象
valid_response = client.post(
    "/users",
    json={
        "username": "小明",
        "age": 20,
        "password": "secret123",
    },
)

invalid_response = client.post(
    "/users",
    json={
        "username": "小明",
        "age": 200,
        "password": "123",
    },
)

print("合法请求：", valid_response.status_code, valid_response.json())
print("非法请求：", invalid_response.status_code)
print(invalid_response.json())

合法请求会进入`create_user()`并返回201。非法请求同时违反年龄范围和密码长度，FastAPI返回422，`create_user()`不会执行。

### 课堂小练习6：构造不同的请求

**目标：**观察Pydantic请求模型如何影响FastAPI响应。

请分别发送下面三份JSON，并记录状态码：

1. 合法数据，但省略可选的`bio`；
2. 缺少必填的`username`；
3. `age`传入字符串`"30"`。

**自检：**前两种应分别得到201和422；第三种默认会被Pydantic解析成整数，因此可以通过。

<details>
<summary>完成后查看示例请求</summary>

```python
client.post("/users", json={"username": "小王", "age": 30, "password": "abcdef"})
client.post("/users", json={"age": 30, "password": "abcdef"})
client.post("/users", json={"username": "小王", "age": "30", "password": "abcdef"})
```

</details>

In [ ]:
# 课堂小练习6：在这里使用client.post()发送三份请求
# 打印每个Response对象的status_code和json()结果


## 8. 为什么请求模型和响应模型要分开？

客户端创建用户时需要提交密码，但服务器绝不能把密码返回给客户端。因此输入和输出不应该使用完全相同的数据模型。

```text
请求模型UserCreate：客户端允许或必须提交什么
响应模型UserPublic：服务器允许返回什么
```


In [ ]:
class UserPublic(BaseModel):
    id: int
    username: str
    age: int
    bio: str | None = None


# app.post(path, status_code=..., response_model=...)：注册POST路由
#   path参数：接口路径
#   status_code参数：成功响应状态码
#   response_model参数：响应必须符合的Pydantic模型
# FastAPI会使用UserPublic验证、转换并过滤最终响应
@app.post("/safe-users", status_code=201, response_model=UserPublic)
def create_safe_user(user: UserCreate) -> dict[str, Any]:
    data = user.model_dump()
    data["id"] = 1
    # data中虽然还有password，但response_model不会把未声明的password返回给客户端
    return data

`response_model=UserPublic`主要产生三种作用：

1. 验证服务器返回的数据是否符合响应契约；
2. 在Swagger/OpenAPI中展示响应结构；
3. 过滤响应模型没有声明的字段，避免返回多余或敏感数据。

请求验证失败通常说明客户端提交的数据不符合契约，FastAPI返回`422`。响应模型验证失败通常说明服务器代码返回了错误结构，是后端实现问题。

### 可运行示例：观察敏感字段被过滤

下面向`/safe-users`提交包含密码的请求，然后检查响应。

In [ ]:
safe_response = client.post(
    "/safe-users",
    json={
        "username": "小赵",
        "age": 25,
        "bio": "喜欢Python",
        "password": "very-secret",
    },
)

safe_data = safe_response.json()
print(safe_response.status_code)
print(safe_data)
print("响应中是否包含password：", "password" in safe_data)

路由函数内部产生的数据包含`password`，但最终JSON响应没有它，因为`UserPublic`没有声明该字段。

### 课堂小练习7：预测响应模型行为

**目标：**理解`response_model`只允许声明过的响应字段。

假设路由返回：

```python
{
    "id": 1,
    "username": "小周",
    "age": 22,
    "bio": None,
    "password": "secret",
    "internal_note": "仅服务器可见",
}
```

在`response_model=UserPublic`下，请先写出你认为客户端最终会收到的字段，再查看上面的`UserPublic`定义自检。

<details>
<summary>完成后查看答案</summary>

客户端只会收到`id`、`username`、`age`和`bio`。`password`与`internal_note`没有在`UserPublic`中声明，因此会被过滤。

</details>

## 9. Swagger中会发生什么变化？

Pydantic模型会被转换成JSON Schema并加入OpenAPI文档。打开`/docs`时，可以看到：

- 请求体必须包含哪些字段；
- 字段类型；
- 哪些字段必填；
- 默认值；
- 长度或数值限制；
- 成功响应的数据结构；
- 验证失败可能返回的422响应。

这就是“数据模型既是验证规则，也是接口文档”的含义。

## 10. 本课整体流程

```text
客户端发送JSON请求体
        ↓
FastAPI解析JSON
        ↓
Pydantic请求模型验证字段
    ├─ 失败 → FastAPI自动返回422，不执行路由函数
    └─ 成功 → 得到模型实例
                    ↓
              执行业务函数
                    ↓
              产生返回数据
                    ↓
Pydantic响应模型验证并过滤
        ↓
FastAPI生成JSON响应
```

## 11. 本课作业：图书创建接口

请完成同目录`app.py`中的`BookCreate`、`BookResponse`和`create_book()`。不要修改类名、函数名、路由路径和测试文件。

### 11.1 客户端请求模型`BookCreate`

| 字段 | 类型 | 是否必填 | 默认值 | 约束 | 来源 |
|---|---|---|---|---|---|
| `title` | 字符串 | 是 | 无 | 长度1至100 | 客户端提交 |
| `price` | 浮点数 | 是 | 无 | 必须严格大于0 | 客户端提交 |
| `in_stock` | 布尔值 | 否 | `True` | 无 | 客户端可提交 |
| `note` | 字符串或`None` | 否 | `None` | 无 | 客户端可提交 |

### 11.2 服务器响应模型`BookResponse`

响应中必须包含：

- 服务器生成的整数`id`，本课固定为`1`；
- 验证后的`title`；
- 验证后的`price`；
- 实际使用的`in_stock`；
- 实际使用的`note`。

`id`不由客户端提交，而是服务器生成。请求体和响应体的数据来源不能混淆。

### 11.3 接口契约

| 方法与路径 | 请求体 | 成功状态码 | 响应模型 |
|---|---|---|---|
| `POST /books` | `BookCreate` | 201 | `BookResponse` |

合法请求示例：

```json
{
  "title": "Python工程实践",
  "price": 59.9
}
```

成功响应中应自动使用`in_stock=true`和`note=null`。缺少必填字段、标题长度不合法、价格不大于0或类型无法解析时，由FastAPI自动返回`422`。

你需要自己决定模型字段如何声明，以及怎样从`BookCreate`实例构造符合`BookResponse`的返回数据。

运行：

```powershell
pytest -v
```

最终目标是`10 passed`。

## 12. 理解题（请把答案写在每题下面）

1. 相比`dict[str, Any]`，Pydantic模型额外提供了哪些能力？

2. 为什么自定义模型类要继承`BaseModel`？创建模型实例时，验证发生在什么时候？

3. `title: str`、`published: bool = False`和`summary: str | None = None`在必填、默认值和可空方面有什么区别？

4. Pydantic模型实例与普通字典有什么区别？`model_dump()`的作用是什么？

5. `Field`中的`min_length`、`max_length`、`gt`和`ge`分别限制什么？

6. FastAPI收到不符合请求模型的数据时，为什么通常不会执行路由处理函数？

7. 为什么创建用户时，请求模型可以包含`password`，响应模型却不应该包含它？

8. `response_model`除了生成文档外，还会对服务器返回的数据做什么？